In [0]:
# ---------------------------------------------------------------------------
# Configuration
# ---------------------------------------------------------------------------
# Schemas of the Adventure Works lakehouse: silver (source) and gold (target).
# Note: this lakehouse has no sales_order_header, so ModifiedDate on the
# sales line is used as the date proxy for the quarter derivation.

source_schema = "adventurework.silver"
gold_schema = "adventurework.gold"

sales_table = f"{source_schema}.fact_sales"      # silver sales order lines
product_table = f"{source_schema}.dim_product"   # silver product dimension
target_table = f"{gold_schema}.sales_kpis_by_category_quarter"

In [0]:
# ---------------------------------------------------------------------------
# Gold KPI table: sales KPIs by product category and quarter
# ---------------------------------------------------------------------------
# Reads silver, joins sales lines to products on ProductID, maps ProductLine
# to readable category names (R = Road, M = Mountain, S = Standard,
# T = Touring, NULL = Other), derives the quarter (yyyy-Qn) from the sales
# line's ModifiedDate, and computes per group: total_revenue, order_count,
# avg_order_value. Writes one gold table with an idempotent full overwrite.

from pyspark.sql.functions import col, sum as _sum, countDistinct, date_format, when

# Read silver sources; select only the columns needed for the KPIs
sales_df = spark.table(sales_table).select(
    "SalesOrderID", "ProductID", "LineTotal", "ModifiedDate"
)
product_df = spark.table(product_table).select(
    "ProductID", "ProductLine"
)

# Join sales lines to products on the product id
joined = sales_df.join(product_df, on="ProductID", how="inner")

# Product category: ProductLine mapped to readable names (NULL -> Other)
joined = joined.withColumn(
    "product_category",
    when(col("ProductLine") == "R", "Road")
    .when(col("ProductLine") == "M", "Mountain")
    .when(col("ProductLine") == "S", "Standard")
    .when(col("ProductLine") == "T", "Touring")
    .otherwise("Other"),
)

# Quarter from ModifiedDate in yyyy-Qn format
joined = joined.withColumn("quarter", date_format("ModifiedDate", "yyyy-'Q'q"))

# One row per product category x quarter
kpis = (
    joined.groupBy("product_category", "quarter")
    .agg(
        _sum("LineTotal").alias("total_revenue"),
        countDistinct("SalesOrderID").alias("order_count"),
    )
    .withColumn("avg_order_value", col("total_revenue") / col("order_count"))
)

row_count = kpis.count()

# Idempotent full overwrite of the gold table
(kpis.write
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(target_table))

print(f"Wrote {row_count} rows to {target_table}")

# Top 10 rows by total_revenue descending
display(
    spark.table(target_table)
    .orderBy(col("total_revenue").desc())
    .limit(10)
)

Wrote 53 rows to adventurework.gold.sales_kpis_by_category_quarter


product_category,quarter,total_revenue,order_count,avg_order_value
Road,2023-Q2,5492956.9679,678,8101.7064423303835
Road,2023-Q3,5190345.6209,694,7478.8841799711816
Road,2024-Q2,4423986.7025,822,5381.9789568126521
Road,2024-Q1,4412414.3437,694,6357.9457402017291
Mountain,2024-Q3,4394856.9081,2381,1845.8029853422932
Mountain,2025-Q1,4318391.5458,2665,1620.4095856660413
Mountain,2024-Q4,4210547.7237,2687,1567.0069682545590
Touring,2025-Q1,4096441.1769,1101,3720.6550198910082
Road,2023-Q4,4042968.5067,670,6034.2813532835821
Road,2024-Q3,4042728.5809,1829,2210.3491420995079
